# 進階範例：整合多個資料來源

整合營收、財務報表、股利及揭露資料進行綜合分析。

In [1]:
from twmops import (
    RevenueFetcher, FinancialFetcher, DividendFetcher,
    DisclosureFetcher, InsidersFetcher
)
import pandas as pd
from datetime import datetime

## 範例 1：完整公司檔案

一次取得單一公司的所有可用資料。

In [2]:
def company_profile(stock_id="2330"):
    """
    Fetch comprehensive profile: revenue, financials, dividends, disclosure, insiders.
    """
    print(f"Fetching comprehensive profile for {stock_id}...\n")
    
    # Parallel fetch all data
    rev_fetcher = RevenueFetcher()
    fin_fetcher = FinancialFetcher()
    div_fetcher = DividendFetcher()
    disc_fetcher = DisclosureFetcher()
    ins_fetcher = InsidersFetcher()
    
    try:
        # Concurrent fetches
        revenue, balance_sheet, dividends, disclosure, pledging = await asyncio.gather(
            rev_fetcher.get_single_revenue(stock_id=stock_id, year=115, month=3),
            fin_fetcher.get_financial_statement(
                stock_id=stock_id, year=115, quarter=1,
                report_type="balance_sheet", format="flat"
            ),
            div_fetcher.get_dividends(stock_id=stock_id, year_start=113, year_end=115),
            disc_fetcher.get_disclosure(stock_id=stock_id, year=115, month=3),
            ins_fetcher.get_share_pledging(stock_id=stock_id, year=115, month=3),
        )
    except Exception as e:
        print(f"Error fetching data: {e}")
        return None
    
    if revenue:
        print(f"Company: {revenue.company_name}")
        print(f"Latest Revenue (Month {revenue.month} {revenue.year}): NT${revenue.revenue:,} thousand")
        print(f"YoY Growth: {revenue.yoy_change:.2f}%")
        print()
    
    # Balance sheet assets
    print(f"Latest Balance Sheet (Q{balance_sheet.quarter} {balance_sheet.year}):")
    print(f"  Total items: {len(balance_sheet.items)}")
    assets = [i for i in balance_sheet.items if 'asset' in i.account_name.lower()]
    if assets:
        print(f"  Sample asset items: {len(assets)}")
    print()
    
    # Dividends
    print(f"Dividend History ({len(dividends.records)} records):")
    for record in dividends.records[-3:]:
        cash = record.cash_dividend or 0
        stock = record.stock_dividend or 0
        print(f"  Year {record.year} Q{record.quarter}: Cash NT${cash:.2f}, Stock NT${stock:.2f}")
    print()
    
    # Related-party transactions
    print(f"Related-Party Transactions:")
    print(f"  Funds lending: {len(disclosure.funds_lending)} records")
    print(f"  Endorsement/Guarantee: {len(disclosure.endorsement_guarantee)} records")
    print()
    
    # Insider pledging
    print(f"Insider Activity:")
    print(f"  Directors/Supervisors with pledging: {len(pledging.details)}")
    if pledging.details:
        valid_ratios = [p.pledge_ratio for p in pledging.details if p.pledge_ratio is not None]
        if valid_ratios:
            avg_ratio = sum(valid_ratios) / len(valid_ratios)
            print(f"  Average pledging ratio: {avg_ratio:.2%}")
    
    return {
        'revenue': revenue,
        'balance_sheet': balance_sheet,
        'dividends': dividends,
        'disclosure': disclosure,
        'pledging': pledging,
    }

profile = company_profile("2330")

SyntaxError: 'await' outside async function (2576755114.py, line 16)

## 範例 2：營收分析與財務指標

結合營收趨勢與損益表資料計算獲利指標。

In [5]:
def revenue_and_profitability(stock_id="2330"):
    """
    Fetch revenue and income statement to calculate profitability metrics.
    """
    rev_fetcher = RevenueFetcher()
    fin_fetcher = FinancialFetcher()
    
    try:
        # Get current period revenue
        revenue = await rev_fetcher.get_single_revenue(stock_id=stock_id, year=115, month=3)
        
        # Get income statement
        income_stmt = await fin_fetcher.get_financial_statement(
            stock_id=stock_id, year=115, quarter=1,
            report_type="income_statement", format="flat"
        )
    except Exception as e:
        print(f"Error fetching data: {e}")
        return None
    
    if revenue:
        print(f"Revenue Analysis for {revenue.company_name}")
        print(f"Month: {revenue.year}-{revenue.month:02d}")
        print(f"Revenue: NT${revenue.revenue:,} thousand")
        print(f"MoM Change: {revenue.mom_change:.2f}%")
        print(f"YoY Change: {revenue.yoy_change:.2f}%")
        print()
    
    print(f"Income Statement Items: {len(income_stmt.items)}")
    print()
    
    # Try to extract key metrics
    metric_keywords = ['revenue', 'profit', 'expense', 'tax', 'earnings']
    sample_items = income_stmt.items[:10]
    for item in sample_items:
        for keyword in metric_keywords:
            if keyword.lower() in item.account_name.lower():
                print(f"{item.account_name}: {item.value}")
                break
    
    return {'revenue': revenue, 'income_statement': income_stmt}

analysis = revenue_and_profitability("2330")

Arelle not available — falling back to lxml parsing (no hierarchy/calculation)
SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi
SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi
No presentation arcs found, falling back to flat facts list


Revenue Analysis for 台積電
Month: 115-03
Revenue: NT$415,191,699 thousand
MoM Change: 30.70%
YoY Change: 45.19%

Income Statement Items: 374

AccumulatedInwardRemittanceOfEarningsAsOfTheEndOfThePeriod: 0


## 範例 3：多公司批量分析

同時分析一組公司進行比較。

In [6]:
def batch_company_analysis(stock_ids=["2330", "3045", "2454"]):
    """
    Analyze multiple companies side-by-side.
    """
    fetcher = RevenueFetcher()
    div_fetcher = DividendFetcher()
    
    print(f"Analyzing {len(stock_ids)} companies...\n")
    
    companies = []
    for stock_id in stock_ids:
        try:
            revenue = fetcher.get_single_revenue(stock_id=stock_id, year=115, month=3)
            dividends = await div_fetcher.get_dividends(stock_id=stock_id, year_start=113, year_end=115)
            
            if revenue:
                latest_dividend = dividends.records[0] if dividends.records else None
                cash_div = latest_dividend.cash_dividend if latest_dividend else 0
                
                companies.append({
                    'Stock ID': stock_id,
                    'Company': revenue.company_name,
                    'Revenue (thousands)': revenue.revenue,
                    'YoY Growth %': revenue.yoy_change,
                    'Cash Dividend': cash_div,
                })
        except Exception as e:
            print(f"Warning: Could not fetch data for {stock_id}: {e}")
            continue
    
    if not companies:
        print("No data retrieved")
        return None
    
    # Create comparison DataFrame
    df = pd.DataFrame(companies)
    print(df.to_string(index=False))
    print()
    
    # Rankings
    print("By Revenue:")
    print(df.nlargest(3, 'Revenue (thousands)')[['Stock ID', 'Company', 'Revenue (thousands)']])
    print()
    
    print("By YoY Growth:")
    print(df.nlargest(3, 'YoY Growth %')[['Stock ID', 'Company', 'YoY Growth %']])
    
    return df

comparison = batch_company_analysis(["2330", "3045", "2454"])

Analyzing 3 companies...



SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi
SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi
SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi
SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi
SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi
SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updatin

Stock ID Company  Revenue (thousands)  YoY Growth %  Cash Dividend
    2330     台積電            415191699         45.19   4.620560e+12
    3045     台灣大             17109290          5.76   6.778500e+04
    2454     聯發科             63219184         12.89   1.709339e+11

By Revenue:
  Stock ID Company  Revenue (thousands)
0     2330     台積電            415191699
2     2454     聯發科             63219184
1     3045     台灣大             17109290

By YoY Growth:
  Stock ID Company  YoY Growth %
0     2330     台積電         45.19
2     2454     聯發科         12.89
1     3045     台灣大          5.76


## 範例 4：股利與報酬分析

分析歷史股利模式與股東報酬。

In [7]:
def dividend_analysis(stock_id="2330"):
    """
    Analyze dividend trends and shareholder returns.
    """
    fetcher = DividendFetcher()
    
    try:
        # Get 5-year history
        dividends = fetcher.get_dividends(
            stock_id=stock_id,
            year_start=110,  # 2021
            year_end=115     # 2026
        )
    except Exception as e:
        print(f"Error fetching dividend data: {e}")
        return None
    
    print(f"Dividend Analysis for {dividends.company_name}")
    print(f"Period: Year {110} - Year {115}\n")
    
    # Create DataFrame
    records = []
    for record in dividends.records:
        cash = record.cash_dividend or 0.0
        stock = record.stock_dividend or 0.0
        total = cash + stock
        records.append({
            'Year': record.year,
            'Quarter': record.quarter,
            'Cash Dividend': f"NT${cash:.2f}",
            'Stock Dividend': f"NT${stock:.2f}",
            'Total Dividend': f"NT${total:.2f}",
        })
    
    if records:
        df = pd.DataFrame(records)
        print(df.to_string(index=False))
        print()
    
    # Summary by year
    print("Total Dividends by Year:")
    yearly = {}
    for record in dividends.records:
        year = record.year
        cash = record.cash_dividend or 0.0
        stock = record.stock_dividend or 0.0
        if year not in yearly:
            yearly[year] = {'cash': 0, 'stock': 0}
        yearly[year]['cash'] += cash
        yearly[year]['stock'] += stock
    
    for year in sorted(yearly.keys()):
        cash = yearly[year]['cash']
        stock = yearly[year]['stock']
        total = cash + stock
        print(f"  Year {year}: Cash NT${cash:.2f}, Stock NT${stock:.2f}, Total NT${total:.2f}")
    
    return dividends

div_data = dividend_analysis("2330")

SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi


Dividend Analysis for 
Period: Year 110 - Year 115

 Year Quarter       Cash Dividend     Stock Dividend      Total Dividend
  115    None NT$4620560103436.00 NT$572479752038.00 NT$5193039855474.00
  114    None NT$4199504001230.00 NT$505743990228.00 NT$4705247991458.00
  114    None NT$3808522634395.00 NT$452301407041.00 NT$4260824041436.00
  114    None NT$3721467292144.00 NT$398273102398.00 NT$4119740394542.00
  114    None NT$3489407823366.00 NT$361564127700.00 NT$3850971951066.00
  113    None NT$3229535042013.00 NT$374679726260.00 NT$3604214768273.00
  113    None NT$3023806292885.00 NT$325257571799.00 NT$3349063864684.00
  113    None NT$2879682286017.00 NT$247845527836.00 NT$3127527813853.00
  113    None NT$2728100821494.00 NT$225484876793.00 NT$2953585698287.00
  112    None NT$2608623649589.00 NT$238712143177.00 NT$2847335792766.00
  112    None NT$2471139126460.00 NT$210999938299.00 NT$2682139064759.00
  112    None NT$2360751031249.00 NT$181799021391.00 NT$2542550052640.00

## 提示

- **非同步並行**: 使用 `asyncio.gather()` 並行取得多個資料來源
- **錯誤處理**: 在個別取得作業中包裝 try/except 以實現穩健的批量操作
- **速率限制**: HTML 客戶端強制每秒 1 個請求；相應規劃批量操作
- **資料整合**: 用 `stock_id` 和報告期間（年/季/月）進行 join 以保持一致
- **民國年轉換**: MOPS 使用民國年（民國年）；用 `year + 1911` 轉換為西元年